In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

print("DeliveryRisk Engine - Baseline Feature Engineering")

DeliveryRisk Engine - Baseline Feature Engineering


In [2]:
DATA_DIR = Path("../data/raw")

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")

print("Data loaded.")

Data loaded.


In [3]:
timestamp_cols = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in timestamp_cols:
    orders[col] = pd.to_datetime(orders[col])

delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

delivered_orders["late_delivery"] = (
    delivered_orders["order_delivered_customer_date"]
    > delivered_orders["order_estimated_delivery_date"]
).astype(int)

print("Delivered orders:", len(delivered_orders))
print(delivered_orders["late_delivery"].value_counts())

Delivered orders: 96478
late_delivery
0    88652
1     7826
Name: count, dtype: int64


In [4]:
order_features = delivered_orders[
    [
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "late_delivery"
    ]
].copy()

order_features["purchase_hour"] = (
    order_features["order_purchase_timestamp"].dt.hour
)

order_features["purchase_dayofweek"] = (
    order_features["order_purchase_timestamp"].dt.dayofweek
)

order_features["purchase_month"] = (
    order_features["order_purchase_timestamp"].dt.month
)

order_features["is_weekend"] = (
    order_features["purchase_dayofweek"] >= 5
).astype(int)

order_features.head()

,order_id,customer_id,order_purchase_timestamp,late_delivery,purchase_hour,purchase_dayofweek,purchase_month,is_weekend
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,0,10,0,10,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,0,20,1,7,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,0,8,2,8,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,0,19,5,11,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,0,21,1,2,0


In [5]:
customer_features = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
].copy()

order_features = order_features.merge(
    customer_features,
    on="customer_id",
    how="left"
)

print(order_features.shape)

(96478, 12)


In [6]:
item_features = (
    order_items
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        avg_item_price=("price", "mean"),
        unique_product_count=("product_id", "nunique"),
        unique_seller_count=("seller_id", "nunique")
    )
    .reset_index()
)

item_features.head()

,order_id,item_count,total_price,total_freight,avg_item_price,unique_product_count,unique_seller_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,58.90,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,239.90,1,1
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,199.00,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,12.99,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,199.90,1,1


In [7]:
order_features = order_features.merge(
    item_features,
    on="order_id",
    how="left"
)

print(order_features.shape)

(96478, 18)


In [8]:
item_seller = order_items[
    ["order_id", "seller_id"]
].merge(
    sellers[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left"
)

seller_features = (
    item_seller
    .groupby("order_id")
    .agg(
        seller_count=("seller_id", "nunique"),
        seller_state_count=("seller_state", "nunique"),
        avg_seller_zip=("seller_zip_code_prefix", "mean")
    )
    .reset_index()
)

order_features = order_features.merge(
    seller_features,
    on="order_id",
    how="left"
)

print(order_features.shape)

(96478, 21)


In [9]:
print("Rows:", len(order_features))
print("Unique orders:", order_features["order_id"].nunique())
print("Columns:", len(order_features.columns))

Rows: 96478
Unique orders: 96478
Columns: 21


In [10]:
OUTPUT_PATH = Path("../data/processed/baseline_features.csv")

order_features.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved to: {OUTPUT_PATH}")

Saved to: ..\data\processed\baseline_features.csv


In [11]:
item_product = order_items[
    ["order_id", "product_id"]
].merge(
    products[
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ],
    on="product_id",
    how="left"
)

product_features = (
    item_product
    .groupby("order_id")
    .agg(
        total_product_weight_g=("product_weight_g", "sum"),
        avg_product_weight_g=("product_weight_g", "mean"),
        avg_product_length_cm=("product_length_cm", "mean"),
        avg_product_height_cm=("product_height_cm", "mean"),
        avg_product_width_cm=("product_width_cm", "mean"),
        unique_product_categories=("product_category_name", "nunique")
    )
    .reset_index()
)

print("Product features created:", product_features.shape)

Product features created: (98666, 7)


In [12]:
order_features = order_features.merge(
    product_features,
    on="order_id",
    how="left"
)

print("Shape:", order_features.shape)
print("Unique orders:", order_features["order_id"].nunique())

Shape: (96478, 27)
Unique orders: 96478


In [13]:
item_seller = order_items[
    ["order_id", "seller_id"]
].merge(
    sellers[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left"
)

seller_features = (
    item_seller
    .groupby("order_id")
    .agg(
        seller_count=("seller_id", "nunique"),
        seller_state_count=("seller_state", "nunique"),
        primary_seller_state=(
            "seller_state",
            lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
        )
    )
    .reset_index()
)

print("Seller features:", seller_features.shape)

Seller features: (98666, 4)


In [14]:
order_features = order_features.merge(
    seller_features,
    on="order_id",
    how="left"
)

print("Shape:", order_features.shape)
print("Unique orders:", order_features["order_id"].nunique())

Shape: (96478, 30)
Unique orders: 96478


In [15]:
shipping = order_items[
    [
        "order_id",
        "shipping_limit_date"
    ]
].copy()

shipping["shipping_limit_date"] = pd.to_datetime(
    shipping["shipping_limit_date"]
)

shipping = shipping.merge(
    order_features[
        [
            "order_id",
            "order_purchase_timestamp"
        ]
    ],
    on="order_id",
    how="inner"
)

shipping["shipping_window_hours"] = (
    shipping["shipping_limit_date"]
    - shipping["order_purchase_timestamp"]
).dt.total_seconds() / 3600

shipping_features = (
    shipping
    .groupby("order_id")
    .agg(
        avg_shipping_window_hours=(
            "shipping_window_hours",
            "mean"
        ),
        min_shipping_window_hours=(
            "shipping_window_hours",
            "min"
        )
    )
    .reset_index()
)

print("Shipping features:", shipping_features.shape)

Shipping features: (96478, 3)


In [16]:
order_features = order_features.merge(
    shipping_features,
    on="order_id",
    how="left"
)

print("Shape:", order_features.shape)
print("Unique orders:", order_features["order_id"].nunique())

Shape: (96478, 32)
Unique orders: 96478


In [17]:
order_features.columns.tolist()

['order_id',
 'customer_id',
 'order_purchase_timestamp',
 'late_delivery',
 'purchase_hour',
 'purchase_dayofweek',
 'purchase_month',
 'is_weekend',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'item_count',
 'total_price',
 'total_freight',
 'avg_item_price',
 'unique_product_count',
 'unique_seller_count',
 'seller_count_x',
 'seller_state_count_x',
 'avg_seller_zip',
 'total_product_weight_g',
 'avg_product_weight_g',
 'avg_product_length_cm',
 'avg_product_height_cm',
 'avg_product_width_cm',
 'unique_product_categories',
 'seller_count_y',
 'seller_state_count_y',
 'primary_seller_state',
 'avg_shipping_window_hours',
 'min_shipping_window_hours']

In [18]:
order_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 96478 entries, 0 to 96477
Data columns (total 32 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   order_id                   96478 non-null  str           
 1   customer_id                96478 non-null  str           
 2   order_purchase_timestamp   96478 non-null  datetime64[us]
 3   late_delivery              96478 non-null  int64         
 4   purchase_hour              96478 non-null  int32         
 5   purchase_dayofweek         96478 non-null  int32         
 6   purchase_month             96478 non-null  int32         
 7   is_weekend                 96478 non-null  int64         
 8   customer_unique_id         96478 non-null  str           
 9   customer_zip_code_prefix   96478 non-null  int64         
 10  customer_city              96478 non-null  str           
 11  customer_state             96478 non-null  str           
 12  item_count     

In [19]:
order_features.isnull().sum().sort_values(ascending=False)

avg_product_height_cm        16
avg_product_width_cm         16
avg_product_length_cm        16
avg_product_weight_g         16
order_id                      0
customer_id                   0
order_purchase_timestamp      0
late_delivery                 0
customer_unique_id            0
customer_zip_code_prefix      0
customer_city                 0
customer_state                0
purchase_hour                 0
purchase_dayofweek            0
purchase_month                0
is_weekend                    0
avg_item_price                0
total_freight                 0
total_price                   0
item_count                    0
seller_state_count_x          0
seller_count_x                0
unique_product_count          0
unique_seller_count           0
avg_seller_zip                0
total_product_weight_g        0
unique_product_categories     0
seller_count_y                0
seller_state_count_y          0
primary_seller_state          0
avg_shipping_window_hours     0
min_ship

In [20]:
order_features["late_delivery"].value_counts()

late_delivery
0    88652
1     7826
Name: count, dtype: int64

In [21]:
order_features["late_delivery"].value_counts(normalize=True) * 100

late_delivery
0    91.888306
1     8.111694
Name: proportion, dtype: float64

In [22]:
id_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

model_data = order_features.drop(
    columns=id_columns
).copy()

print("Model data shape:", model_data.shape)

Model data shape: (96478, 29)


In [23]:
categorical_columns = model_data.select_dtypes(
    include=["object"]
).columns.tolist()

print(categorical_columns)

['customer_city', 'customer_state', 'primary_seller_state']


C:\Users\anant\AppData\Local\Temp\ipykernel_26908\3258765731.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = model_data.select_dtypes(


In [24]:
numerical_columns = model_data.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print(numerical_columns)

['late_delivery', 'is_weekend', 'customer_zip_code_prefix', 'item_count', 'total_price', 'total_freight', 'avg_item_price', 'unique_product_count', 'unique_seller_count', 'seller_count_x', 'seller_state_count_x', 'avg_seller_zip', 'total_product_weight_g', 'avg_product_weight_g', 'avg_product_length_cm', 'avg_product_height_cm', 'avg_product_width_cm', 'unique_product_categories', 'seller_count_y', 'seller_state_count_y', 'avg_shipping_window_hours', 'min_shipping_window_hours']


In [25]:
model_data = model_data.drop(
    columns=["avg_seller_zip"],
    errors="ignore"
)

In [26]:
print(model_data.dtypes)

order_purchase_timestamp     datetime64[us]
late_delivery                         int64
purchase_hour                         int32
purchase_dayofweek                    int32
purchase_month                        int32
is_weekend                            int64
customer_zip_code_prefix              int64
customer_city                           str
customer_state                          str
item_count                            int64
total_price                         float64
total_freight                       float64
avg_item_price                      float64
unique_product_count                  int64
unique_seller_count                   int64
seller_count_x                        int64
seller_state_count_x                  int64
total_product_weight_g              float64
avg_product_weight_g                float64
avg_product_length_cm               float64
avg_product_height_cm               float64
avg_product_width_cm                float64
unique_product_categories       

In [27]:
print("Rows:", len(model_data))
print("Columns:", len(model_data.columns))

Rows: 96478
Columns: 28


In [28]:
model_data = order_features.copy()

# Sort chronologically BEFORE removing the timestamp
model_data = model_data.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

# Remove identifiers and fields that should not go directly into the model
model_data = model_data.drop(
    columns=[
        "order_id",
        "customer_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "avg_seller_zip"
    ],
    errors="ignore"
)

print("Rows:", len(model_data))
print("Columns:", len(model_data.columns))
print("\nRemaining columns:")
print(model_data.columns.tolist())

Rows: 96478
Columns: 27

Remaining columns:
['late_delivery', 'purchase_hour', 'purchase_dayofweek', 'purchase_month', 'is_weekend', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'total_price', 'total_freight', 'avg_item_price', 'unique_product_count', 'unique_seller_count', 'seller_count_x', 'seller_state_count_x', 'total_product_weight_g', 'avg_product_weight_g', 'avg_product_length_cm', 'avg_product_height_cm', 'avg_product_width_cm', 'unique_product_categories', 'seller_count_y', 'seller_state_count_y', 'primary_seller_state', 'avg_shipping_window_hours', 'min_shipping_window_hours']


In [29]:
model_data.isnull().sum().sort_values(ascending=False)

avg_product_width_cm         16
avg_product_length_cm        16
avg_product_weight_g         16
avg_product_height_cm        16
is_weekend                    0
customer_zip_code_prefix      0
customer_city                 0
purchase_month                0
late_delivery                 0
purchase_hour                 0
purchase_dayofweek            0
total_freight                 0
total_price                   0
item_count                    0
customer_state                0
seller_count_x                0
unique_seller_count           0
avg_item_price                0
unique_product_count          0
seller_state_count_x          0
total_product_weight_g        0
unique_product_categories     0
seller_count_y                0
seller_state_count_y          0
primary_seller_state          0
avg_shipping_window_hours     0
min_shipping_window_hours     0
dtype: int64

In [30]:
for i, col in enumerate(model_data.columns):
    print(i, col)

0 late_delivery
1 purchase_hour
2 purchase_dayofweek
3 purchase_month
4 is_weekend
5 customer_zip_code_prefix
6 customer_city
7 customer_state
8 item_count
9 total_price
10 total_freight
11 avg_item_price
12 unique_product_count
13 unique_seller_count
14 seller_count_x
15 seller_state_count_x
16 total_product_weight_g
17 avg_product_weight_g
18 avg_product_length_cm
19 avg_product_height_cm
20 avg_product_width_cm
21 unique_product_categories
22 seller_count_y
23 seller_state_count_y
24 primary_seller_state
25 avg_shipping_window_hours
26 min_shipping_window_hours


In [31]:
# ============================================================
# CLEAN BASELINE FEATURE SET
# ============================================================

baseline_columns = [
    # Target
    "late_delivery",

    # Time
    "purchase_hour",
    "purchase_dayofweek",
    "purchase_month",
    "is_weekend",

    # Customer
    "customer_state",

    # Order / item features
    "item_count",
    "total_price",
    "total_freight",
    "avg_item_price",
    "unique_product_count",
    "unique_seller_count",

    # Product features
    "total_product_weight_g",
    "avg_product_weight_g",
    "avg_product_length_cm",
    "avg_product_height_cm",
    "avg_product_width_cm",
    "unique_product_categories",

    # Seller features
    "seller_count_x",
    "seller_state_count_x",
    "primary_seller_state",

    # Shipping
    "avg_shipping_window_hours",
    "min_shipping_window_hours",
]

# Check that every requested column exists
missing_columns = [
    col for col in baseline_columns
    if col not in order_features.columns
]

print("Missing requested columns:", missing_columns)

# Create clean dataset
baseline_data = order_features[baseline_columns].copy()

print("Baseline shape:", baseline_data.shape)
print("Unique rows:", len(baseline_data))
print("\nColumns:")
print(baseline_data.columns.tolist())

Missing requested columns: []
Baseline shape: (96478, 23)
Unique rows: 96478

Columns:
['late_delivery', 'purchase_hour', 'purchase_dayofweek', 'purchase_month', 'is_weekend', 'customer_state', 'item_count', 'total_price', 'total_freight', 'avg_item_price', 'unique_product_count', 'unique_seller_count', 'total_product_weight_g', 'avg_product_weight_g', 'avg_product_length_cm', 'avg_product_height_cm', 'avg_product_width_cm', 'unique_product_categories', 'seller_count_x', 'seller_state_count_x', 'primary_seller_state', 'avg_shipping_window_hours', 'min_shipping_window_hours']


In [32]:
baseline_data.isnull().sum().sort_values(ascending=False)

avg_product_width_cm         16
avg_product_length_cm        16
avg_product_weight_g         16
avg_product_height_cm        16
late_delivery                 0
purchase_hour                 0
purchase_dayofweek            0
item_count                    0
customer_state                0
is_weekend                    0
purchase_month                0
unique_product_count          0
avg_item_price                0
total_price                   0
total_freight                 0
unique_seller_count           0
total_product_weight_g        0
unique_product_categories     0
seller_count_x                0
seller_state_count_x          0
primary_seller_state          0
avg_shipping_window_hours     0
min_shipping_window_hours     0
dtype: int64

In [37]:
if len(baseline_data) != len(order_features):
    raise ValueError(
        f"Row mismatch: baseline_data={len(baseline_data)}, "
        f"order_features={len(order_features)}"
    )

model_base = baseline_data.copy()

model_base.insert(
    0,
    "order_id",
    order_features["order_id"].to_numpy()
)

model_base.insert(
    1,
    "order_purchase_timestamp",
    pd.to_datetime(
        order_features["order_purchase_timestamp"]
    ).to_numpy()
)

model_base = (
    model_base
    .sort_values("order_purchase_timestamp")
    .reset_index(drop=True)
)

print("Rows:", len(model_base))
print("Unique orders:", model_base["order_id"].nunique())

print("\nTime range:")
print(
    "Start:",
    model_base["order_purchase_timestamp"].min()
)
print(
    "End:",
    model_base["order_purchase_timestamp"].max()
)


Rows: 96478
Unique orders: 96478

Time range:
Start: 2016-09-15 12:16:38
End: 2018-08-29 15:00:37


In [38]:
n = len(model_base)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = model_base.iloc[:train_end].copy()
val_data = model_base.iloc[train_end:val_end].copy()
test_data = model_base.iloc[val_end:].copy()

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

print("\nTime ranges")

print(
    "Train:",
    train_data["order_purchase_timestamp"].min(),
    "→",
    train_data["order_purchase_timestamp"].max()
)

print(
    "Validation:",
    val_data["order_purchase_timestamp"].min(),
    "→",
    val_data["order_purchase_timestamp"].max()
)

print(
    "Test:",
    test_data["order_purchase_timestamp"].min(),
    "→",
    test_data["order_purchase_timestamp"].max()
)

Train: (67534, 25)
Validation: (14472, 25)
Test: (14472, 25)

Time ranges
Train: 2016-09-15 12:16:38 → 2018-04-15 20:22:16
Validation: 2018-04-15 20:23:05 → 2018-06-21 08:46:52
Test: 2018-06-21 08:48:47 → 2018-08-29 15:00:37


In [39]:
print(
    "Train end < Validation start:",
    train_data["order_purchase_timestamp"].max()
    < val_data["order_purchase_timestamp"].min()
)

print(
    "Validation end < Test start:",
    val_data["order_purchase_timestamp"].max()
    < test_data["order_purchase_timestamp"].min()
)

Train end < Validation start: True
Validation end < Test start: True


In [40]:
DROP_COLUMNS = [
    "late_delivery",
    "order_id",
    "order_purchase_timestamp"
]

X_train = train_data.drop(columns=DROP_COLUMNS)
y_train = train_data["late_delivery"]

X_val = val_data.drop(columns=DROP_COLUMNS)
y_val = val_data["late_delivery"]

X_test = test_data.drop(columns=DROP_COLUMNS)
y_test = test_data["late_delivery"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (67534, 22)
y_train: (67534,)
X_val: (14472, 22)
y_val: (14472,)
X_test: (14472, 22)
y_test: (14472,)


In [41]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Identify feature types
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:", numeric_features)
print("\nCategorical features:", categorical_features)

# Numeric preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combined preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Logistic Regression baseline
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

print("\nLogistic Regression trained successfully.")

C:\Users\anant\AppData\Local\Temp\ipykernel_26908\2335594644.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


Numeric features: ['is_weekend', 'item_count', 'total_price', 'total_freight', 'avg_item_price', 'unique_product_count', 'unique_seller_count', 'total_product_weight_g', 'avg_product_weight_g', 'avg_product_length_cm', 'avg_product_height_cm', 'avg_product_width_cm', 'unique_product_categories', 'seller_count_x', 'seller_state_count_x', 'avg_shipping_window_hours', 'min_shipping_window_hours']

Categorical features: ['customer_state', 'primary_seller_state']

Logistic Regression trained successfully.


In [42]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

val_predictions = logistic_model.predict(X_val)
val_probabilities = logistic_model.predict_proba(X_val)[:, 1]

print("Logistic Regression — Validation Results")
print("Accuracy :", accuracy_score(y_val, val_predictions))
print("Precision:", precision_score(y_val, val_predictions, zero_division=0))
print("Recall   :", recall_score(y_val, val_predictions, zero_division=0))
print("F1       :", f1_score(y_val, val_predictions, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_val, val_probabilities))
print("PR-AUC   :", average_precision_score(y_val, val_probabilities))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_predictions))

Logistic Regression — Validation Results
Accuracy : 0.6701907131011609
Precision: 0.0855781185246581
Recall   : 0.5342820181112549
F1       : 0.14752634398999823
ROC-AUC  : 0.6490242486609394
PR-AUC   : 0.0915251190728249

Confusion Matrix:
[[9286 4413]
 [ 360  413]]


In [43]:
from sklearn.dummy import DummyClassifier

dummy_model = DummyClassifier(
    strategy="most_frequent"
)

dummy_model.fit(X_train, y_train)

dummy_predictions = dummy_model.predict(X_val)

print("Dummy Baseline")
print("Accuracy :", accuracy_score(y_val, dummy_predictions))
print("Precision:", precision_score(y_val, dummy_predictions, zero_division=0))
print("Recall   :", recall_score(y_val, dummy_predictions, zero_division=0))
print("F1       :", f1_score(y_val, dummy_predictions, zero_division=0))

Dummy Baseline
Accuracy : 0.9465865118850193
Precision: 0.0
Recall   : 0.0
F1       : 0.0


In [44]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [45]:
rf_val_predictions = rf_model.predict(X_val)
rf_val_probabilities = rf_model.predict_proba(X_val)[:, 1]

print("Random Forest — Validation Results")
print("Accuracy :", accuracy_score(y_val, rf_val_predictions))
print("Precision:", precision_score(y_val, rf_val_predictions, zero_division=0))
print("Recall   :", recall_score(y_val, rf_val_predictions, zero_division=0))
print("F1       :", f1_score(y_val, rf_val_predictions, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_val, rf_val_probabilities))
print("PR-AUC   :", average_precision_score(y_val, rf_val_probabilities))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, rf_val_predictions))

Random Forest — Validation Results
Accuracy : 0.6900912106135987
Precision: 0.07722095671981777
Recall   : 0.4385510996119017
F1       : 0.13131900058105753
ROC-AUC  : 0.5632270114994088
PR-AUC   : 0.06989450483982934

Confusion Matrix:
[[9648 4051]
 [ 434  339]]


In [46]:
results = pd.DataFrame([
    {
        "model": "Dummy",
        "accuracy": accuracy_score(y_val, dummy_predictions),
        "precision": precision_score(y_val, dummy_predictions, zero_division=0),
        "recall": recall_score(y_val, dummy_predictions, zero_division=0),
        "f1": f1_score(y_val, dummy_predictions, zero_division=0)
    },
    {
        "model": "Logistic Regression",
        "accuracy": accuracy_score(y_val, val_predictions),
        "precision": precision_score(y_val, val_predictions, zero_division=0),
        "recall": recall_score(y_val, val_predictions, zero_division=0),
        "f1": f1_score(y_val, val_predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_val, val_probabilities),
        "pr_auc": average_precision_score(y_val, val_probabilities)
    },
    {
        "model": "Random Forest",
        "accuracy": accuracy_score(y_val, rf_val_predictions),
        "precision": precision_score(y_val, rf_val_predictions, zero_division=0),
        "recall": recall_score(y_val, rf_val_predictions, zero_division=0),
        "f1": f1_score(y_val, rf_val_predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_val, rf_val_probabilities),
        "pr_auc": average_precision_score(y_val, rf_val_probabilities)
    }
])

results

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Dummy,0.946587,0.000000,0.000000,0.000000,NaN,NaN
1,Logistic Regression,0.670191,0.085578,0.534282,0.147526,0.649024,0.091525
2,Random Forest,0.690091,0.077221,0.438551,0.131319,0.563227,0.069895


In [47]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.10, 0.91, 0.05)

threshold_results = []

for threshold in thresholds:
    predictions = (
        val_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "flag_rate": predictions.mean()
    })

threshold_results = pd.DataFrame(threshold_results)

threshold_results

,threshold,precision,recall,f1,flag_rate
0,0.10,0.053625,1.000000,0.101791,0.996061
1,0.15,0.054037,1.000000,0.102533,0.988460
2,0.20,0.054477,0.998706,0.103319,0.979201
3,0.25,0.055130,0.996119,0.104478,0.965105
4,0.30,0.057080,0.971539,0.107825,0.909135
5,0.35,0.058622,0.927555,0.110274,0.845149
6,0.40,0.070041,0.778784,0.128523,0.593905
7,0.45,0.080518,0.595084,0.141844,0.394762
8,0.50,0.085578,0.534282,0.147526,0.333472
9,0.55,0.094948,0.483829,0.158744,0.272181


In [48]:
threshold_results.sort_values(
    "f1",
    ascending=False
).head(10)

,threshold,precision,recall,f1,flag_rate
10,0.60,0.100869,0.420440,0.162703,0.222637
9,0.55,0.094948,0.483829,0.158744,0.272181
11,0.65,0.100735,0.301423,0.151005,0.159826
8,0.50,0.085578,0.534282,0.147526,0.333472
7,0.45,0.080518,0.595084,0.141844,0.394762
6,0.40,0.070041,0.778784,0.128523,0.593905
12,0.70,0.114108,0.142303,0.126655,0.066611
5,0.35,0.058622,0.927555,0.110274,0.845149
4,0.30,0.057080,0.971539,0.107825,0.909135
13,0.75,0.156962,0.080207,0.106164,0.027294


In [49]:
threshold_results.sort_values(
    "recall",
    ascending=False
).head(10)

,threshold,precision,recall,f1,flag_rate
0,0.10,0.053625,1.000000,0.101791,0.996061
1,0.15,0.054037,1.000000,0.102533,0.988460
2,0.20,0.054477,0.998706,0.103319,0.979201
3,0.25,0.055130,0.996119,0.104478,0.965105
4,0.30,0.057080,0.971539,0.107825,0.909135
5,0.35,0.058622,0.927555,0.110274,0.845149
6,0.40,0.070041,0.778784,0.128523,0.593905
7,0.45,0.080518,0.595084,0.141844,0.394762
8,0.50,0.085578,0.534282,0.147526,0.333472
9,0.55,0.094948,0.483829,0.158744,0.272181


In [50]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/raw")

geolocation = pd.read_csv(
    DATA_DIR / "olist_geolocation_dataset.csv"
)

customers_geo = pd.read_csv(
    DATA_DIR / "olist_customers_dataset.csv"
)

sellers_geo = pd.read_csv(
    DATA_DIR / "olist_sellers_dataset.csv"
)

order_items_geo = pd.read_csv(
    DATA_DIR / "olist_order_items_dataset.csv"
)

print("Geolocation:", geolocation.shape)

Geolocation: (1000163, 5)


In [51]:
geo_lookup = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        latitude=("geolocation_lat", "mean"),
        longitude=("geolocation_lng", "mean")
    )
    .reset_index()
)

print("ZIP coordinate entries:", len(geo_lookup))

ZIP coordinate entries: 19015


In [53]:
orders_geo = pd.read_csv(
    DATA_DIR / "olist_orders_dataset.csv",
    usecols=["order_id", "customer_id"]
)

order_seller_geo = (
    order_items_geo[
        ["order_id", "seller_id"]
    ]
    .drop_duplicates()
    .merge(
        orders_geo,
        on="order_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        customers_geo[
            [
                "customer_id",
                "customer_zip_code_prefix"
            ]
        ],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        sellers_geo[
            [
                "seller_id",
                "seller_zip_code_prefix"
            ]
        ],
        on="seller_id",
        how="left",
        validate="many_to_one"
    )
)

print("Order-seller pairs:", len(order_seller_geo))

Order-seller pairs: 100010


In [54]:
order_seller_geo = order_seller_geo.merge(
    geo_lookup,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
).rename(
    columns={
        "latitude": "customer_lat",
        "longitude": "customer_lng"
    }
).drop(
    columns=["geolocation_zip_code_prefix"]
)

order_seller_geo = order_seller_geo.merge(
    geo_lookup,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
).rename(
    columns={
        "latitude": "seller_lat",
        "longitude": "seller_lng"
    }
).drop(
    columns=["geolocation_zip_code_prefix"]
)

In [55]:
from numpy import radians, sin, cos, sqrt, arctan2

lat_customer = radians(order_seller_geo["customer_lat"])
lon_customer = radians(order_seller_geo["customer_lng"])

lat_seller = radians(order_seller_geo["seller_lat"])
lon_seller = radians(order_seller_geo["seller_lng"])

dlat = lat_seller - lat_customer
dlon = lon_seller - lon_customer

a = (
    sin(dlat / 2) ** 2
    + cos(lat_customer)
    * cos(lat_seller)
    * sin(dlon / 2) ** 2
)

order_seller_geo["distance_km"] = (
    2
    * 6371
    * arctan2(
        sqrt(a),
        sqrt(1 - a)
    )
)

In [56]:
geo_features = (
    order_seller_geo
    .groupby("order_id")
    .agg(
        avg_seller_customer_distance_km=(
            "distance_km",
            "mean"
        ),
        min_seller_customer_distance_km=(
            "distance_km",
            "min"
        ),
        max_seller_customer_distance_km=(
            "distance_km",
            "max"
        )
    )
    .reset_index()
)

print("Geography features:", geo_features.shape)

Geography features: (98666, 4)


In [57]:
geo_features[
    [
        "avg_seller_customer_distance_km",
        "min_seller_customer_distance_km",
        "max_seller_customer_distance_km"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
avg_seller_customer_distance_km,98177.0,601.687892,594.927039,0.0,187.576577,433.814287,798.331635,8677.911622
min_seller_customer_distance_km,98177.0,600.389087,595.027694,0.0,182.788275,431.839596,797.600664,8677.911622
max_seller_customer_distance_km,98177.0,602.992097,595.408565,0.0,188.984835,435.435291,801.166405,8677.911622


In [58]:
print(
    "Missing average distance:",
    geo_features["avg_seller_customer_distance_km"].isna().sum()
)

print(
    "Negative distances:",
    (geo_features["avg_seller_customer_distance_km"] < 0).sum()
)

Missing average distance: 489
Negative distances: 0


In [59]:
geo_model_base = model_base.merge(
    geo_features,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(geo_model_base))
print("Unique orders:", geo_model_base["order_id"].nunique())
print("Columns:", len(geo_model_base.columns))

Rows: 96478
Unique orders: 96478
Columns: 28


In [60]:
geo_model_base[
    [
        "avg_seller_customer_distance_km",
        "min_seller_customer_distance_km",
        "max_seller_customer_distance_km"
    ]
].isna().sum()

avg_seller_customer_distance_km    476
min_seller_customer_distance_km    476
max_seller_customer_distance_km    476
dtype: int64

In [61]:
n = len(geo_model_base)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

geo_train = geo_model_base.iloc[:train_end].copy()
geo_val = geo_model_base.iloc[train_end:val_end].copy()
geo_test = geo_model_base.iloc[val_end:].copy()

print("Train:", geo_train.shape)
print("Validation:", geo_val.shape)
print("Test:", geo_test.shape)

Train: (67534, 28)
Validation: (14472, 28)
Test: (14472, 28)


In [62]:
DROP_COLUMNS = [
    "late_delivery",
    "order_id",
    "order_purchase_timestamp"
]

X_geo_train = geo_train.drop(columns=DROP_COLUMNS)
y_geo_train = geo_train["late_delivery"]

X_geo_val = geo_val.drop(columns=DROP_COLUMNS)
y_geo_val = geo_val["late_delivery"]

X_geo_test = geo_test.drop(columns=DROP_COLUMNS)
y_geo_test = geo_test["late_delivery"]

print("X_geo_train:", X_geo_train.shape)
print("X_geo_val:", X_geo_val.shape)

X_geo_train: (67534, 25)
X_geo_val: (14472, 25)


In [63]:
numeric_geo = X_geo_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_geo = X_geo_train.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_geo_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_geo_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

geo_preprocessor = ColumnTransformer([
    ("numeric", numeric_geo_pipeline, numeric_geo),
    ("categorical", categorical_geo_pipeline, categorical_geo)
])

geo_logistic_model = Pipeline([
    ("preprocessor", geo_preprocessor),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

geo_logistic_model.fit(
    X_geo_train,
    y_geo_train
)

print("Geography-enhanced Logistic Regression trained.")

C:\Users\anant\AppData\Local\Temp\ipykernel_26908\1872893854.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_geo = X_geo_train.select_dtypes(


Geography-enhanced Logistic Regression trained.


In [64]:
geo_val_predictions = geo_logistic_model.predict(X_geo_val)
geo_val_probabilities = (
    geo_logistic_model.predict_proba(X_geo_val)[:, 1]
)

print("Geography-enhanced Logistic Regression")
print(
    "Accuracy :",
    accuracy_score(y_geo_val, geo_val_predictions)
)
print(
    "Precision:",
    precision_score(
        y_geo_val,
        geo_val_predictions,
        zero_division=0
    )
)
print(
    "Recall   :",
    recall_score(
        y_geo_val,
        geo_val_predictions,
        zero_division=0
    )
)
print(
    "F1       :",
    f1_score(
        y_geo_val,
        geo_val_predictions,
        zero_division=0
    )
)
print(
    "ROC-AUC  :",
    roc_auc_score(
        y_geo_val,
        geo_val_probabilities
    )
)
print(
    "PR-AUC   :",
    average_precision_score(
        y_geo_val,
        geo_val_probabilities
    )
)

Geography-enhanced Logistic Regression
Accuracy : 0.6722636815920398
Precision: 0.08559498956158663
Recall   : 0.5304010349288486
F1       : 0.14740248067589432
ROC-AUC  : 0.6479123744124626
PR-AUC   : 0.09370928750762


In [65]:
delay_analysis = delivered_orders.copy()

delay_analysis["delivery_delay_days"] = (
    delay_analysis["order_delivered_customer_date"]
    - delay_analysis["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 3600)

print(delay_analysis["delivery_delay_days"].describe())

print("\nPercent late:")
print(
    (delay_analysis["delivery_delay_days"] > 0).mean() * 100
)

count    96470.000000
mean       -11.178126
std         10.184354
min       -146.016123
25%        -16.244065
50%        -11.948102
75%         -6.389815
max        188.975081
Name: delivery_delay_days, dtype: float64

Percent late:
8.111693857667031


In [66]:
delay_analysis["delay_bucket"] = pd.cut(
    delay_analysis["delivery_delay_days"],
    bins=[-np.inf, -1, 0, 3, 7, 14, np.inf],
    labels=[
        "Early >1 day",
        "On time",
        "1-3 days late",
        "4-7 days late",
        "8-14 days late",
        ">14 days late"
    ]
)

print(
    delay_analysis["delay_bucket"]
    .value_counts()
    .sort_index()
)

delay_bucket
Early >1 day      87182
On time            1462
1-3 days late      2662
4-7 days late      1819
8-14 days late     1790
>14 days late      1555
Name: count, dtype: int64


In [67]:
late_only = delay_analysis[
    delay_analysis["delivery_delay_days"] > 0
].copy()

print("Late orders:", len(late_only))

print("\nDelay statistics for late orders:")
print(
    late_only["delivery_delay_days"].describe()
)

Late orders: 7826

Delay statistics for late orders:
count    7826.000000
mean        9.551776
std        13.952540
min         0.002500
25%         1.867179
50%         5.806481
75%        11.820978
max       188.975081
Name: delivery_delay_days, dtype: float64


In [68]:
severity_counts = pd.cut(
    late_only["delivery_delay_days"],
    bins=[0, 3, 7, 14, np.inf],
    labels=[
        "1-3 days",
        "4-7 days",
        "8-14 days",
        ">14 days"
    ],
    include_lowest=True
).value_counts().sort_index()

print("\nLate-delivery severity:")
print(severity_counts)


Late-delivery severity:
delivery_delay_days
1-3 days     2662
4-7 days     1819
8-14 days    1790
>14 days     1555
Name: count, dtype: int64


In [69]:
print("\nPercentage of late deliveries by severity:")
print(
    severity_counts
    .div(severity_counts.sum())
    .mul(100)
)


Percentage of late deliveries by severity:
delivery_delay_days
1-3 days     34.014822
4-7 days     23.243036
8-14 days    22.872476
>14 days     19.869665
Name: count, dtype: float64


In [71]:
late_only = delay_analysis[
    delay_analysis["delivery_delay_days"] > 0
].copy()

print("Late orders:", len(late_only))

print("\nDelay statistics for late orders:")
print(late_only["delivery_delay_days"].describe())

Late orders: 7826

Delay statistics for late orders:
count    7826.000000
mean        9.551776
std        13.952540
min         0.002500
25%         1.867179
50%         5.806481
75%        11.820978
max       188.975081
Name: delivery_delay_days, dtype: float64


In [72]:
severity_counts = pd.cut(
    late_only["delivery_delay_days"],
    bins=[0, 3, 7, 14, np.inf],
    labels=[
        "1-3 days",
        "4-7 days",
        "8-14 days",
        ">14 days"
    ],
    include_lowest=True
).value_counts().sort_index()

print("\nLate-delivery severity:")
print(severity_counts)


Late-delivery severity:
delivery_delay_days
1-3 days     2662
4-7 days     1819
8-14 days    1790
>14 days     1555
Name: count, dtype: int64


In [73]:
print("\nPercentage of late deliveries by severity:")
print(
    severity_counts
    .div(severity_counts.sum())
    .mul(100)
)


Percentage of late deliveries by severity:
delivery_delay_days
1-3 days     34.014822
4-7 days     23.243036
8-14 days    22.872476
>14 days     19.869665
Name: count, dtype: float64


In [74]:
late_only["delay_severity"] = pd.cut(
    late_only["delivery_delay_days"],
    bins=[0, 3, 7, 14, np.inf],
    labels=[
        "low",
        "moderate",
        "high",
        "critical"
    ],
    include_lowest=True
)

print(
    late_only[
        [
            "order_id",
            "delivery_delay_days",
            "delay_severity"
        ]
    ].head(20)
)

                             order_id  delivery_delay_days delay_severity
20   203096f03d82e0dffbc41ebc2e2bcfb7            11.933171           high
25   fbf9ac61453ac646ce8ad9783d7d0af6             9.919375           high
35   8563039e855156e48fccee4d611a3196             0.041262            low
41   6ea2f835b4556291ffdc53fa0b3b95e8             7.791238           high
57   66e4624ae69e7dc89bd50222b59f581f             1.561644            low
58   a685d016c8a26f71a0bb67821070e398             7.567546           high
97   6a0a8bfbbe700284feb0845d95e0867f            17.821528       critical
102  a5474c0071dd5d1074e12d417078bbd0             1.811655            low
110  9d531c565e28c3e0d756192f84d8731f            32.901991       critical
115  8fc207e94fa91a7649c5a5dab690272a            32.571088       critical
143  33a3edb84b9df4cb49546859b990ac6d             6.002697       moderate
152  1d067305b599c1e0dceb3864056ea527             0.911528            low
196  3f849648ffbabb0562c7668f212e3e88 

In [75]:
print(
    late_only["delay_severity"]
    .value_counts()
    .sort_index()
)

delay_severity
low         2662
moderate    1819
high        1790
critical    1555
Name: count, dtype: int64


In [76]:
late_only = delay_analysis[
    delay_analysis["delivery_delay_days"] > 0
].copy()

print("Late orders:", len(late_only))

print("\nDelay statistics:")
print(late_only["delivery_delay_days"].describe())

Late orders: 7826

Delay statistics:
count    7826.000000
mean        9.551776
std        13.952540
min         0.002500
25%         1.867179
50%         5.806481
75%        11.820978
max       188.975081
Name: delivery_delay_days, dtype: float64


In [77]:
late_only["delay_severity"] = pd.cut(
    late_only["delivery_delay_days"],
    bins=[0, 3, 7, 14, np.inf],
    labels=[
        "low",
        "moderate",
        "high",
        "critical"
    ],
    include_lowest=True
)

print(
    late_only["delay_severity"]
    .value_counts()
    .sort_index()
)

delay_severity
low         2662
moderate    1819
high        1790
critical    1555
Name: count, dtype: int64


In [78]:
late_ids = set(
    late_only["order_id"]
)

stage2_data = model_base[
    model_base["order_id"].isin(late_ids)
].copy()

stage2_targets = late_only[
    [
        "order_id",
        "delivery_delay_days"
    ]
].copy()

stage2_data = stage2_data.merge(
    stage2_targets,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

print("Stage-2 rows:", len(stage2_data))
print(
    "Unique orders:",
    stage2_data["order_id"].nunique()
)
print(
    "Target mean:",
    stage2_data["delivery_delay_days"].mean()
)
print(
    "Target median:",
    stage2_data["delivery_delay_days"].median()
)

Stage-2 rows: 7826
Unique orders: 7826
Target mean: 9.551775917880569
Target median: 5.806481481481482


In [79]:
stage2_data = (
    stage2_data
    .sort_values("order_purchase_timestamp")
    .reset_index(drop=True)
)

In [80]:
stage2_data["log_delay"] = np.log1p(
    stage2_data["delivery_delay_days"]
)

print(stage2_data["log_delay"].describe())

count    7826.000000
mean        1.881571
std         0.949924
min         0.002497
25%         1.053329
50%         1.917875
75%         2.551083
max         5.246893
Name: log_delay, dtype: float64


In [81]:
n2 = len(stage2_data)

train_end_2 = int(n2 * 0.70)
val_end_2 = int(n2 * 0.85)

stage2_train = stage2_data.iloc[:train_end_2].copy()
stage2_val = stage2_data.iloc[train_end_2:val_end_2].copy()
stage2_test = stage2_data.iloc[val_end_2:].copy()

print("Train:", stage2_train.shape)
print("Validation:", stage2_val.shape)
print("Test:", stage2_test.shape)

Train: (5478, 27)
Validation: (1174, 27)
Test: (1174, 27)


In [82]:
DROP_STAGE2 = [
    "order_id",
    "order_purchase_timestamp",
    "delivery_delay_days",
    "log_delay"
]

X2_train = stage2_train.drop(columns=DROP_STAGE2)
y2_train = stage2_train["log_delay"]

X2_val = stage2_val.drop(columns=DROP_STAGE2)
y2_val = stage2_val["log_delay"]

X2_test = stage2_test.drop(columns=DROP_STAGE2)
y2_test = stage2_test["log_delay"]

print("X2_train:", X2_train.shape)
print("y2_train:", y2_train.shape)

X2_train: (5478, 23)
y2_train: (5478,)


In [84]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

dummy_regressor = DummyRegressor(
    strategy="median"
)

dummy_regressor.fit(
    X2_train,
    y2_train
)

dummy_log_pred = dummy_regressor.predict(X2_val)

print("Dummy Regression Baseline")
print("MAE (log):",
      mean_absolute_error(y2_val, dummy_log_pred))
print("RMSE (log):",
      np.sqrt(mean_squared_error(y2_val, dummy_log_pred)))
print("R²:",
      r2_score(y2_val, dummy_log_pred))

Dummy Regression Baseline
MAE (log): 0.7470268579812614
RMSE (log): 0.9126487126809422
R²: -0.06847542953657082


In [85]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_features_2 = X2_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_2 = X2_train.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_pipeline_2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline_2 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_2 = ColumnTransformer([
    ("numeric", numeric_pipeline_2, numeric_features_2),
    ("categorical", categorical_pipeline_2, categorical_features_2)
])

rf_regressor = Pipeline([
    ("preprocessor", preprocessor_2),
    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_regressor.fit(
    X2_train,
    y2_train
)

print("Stage-2 Random Forest trained successfully.")

C:\Users\anant\AppData\Local\Temp\ipykernel_26908\2705274233.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features_2 = X2_train.select_dtypes(


Stage-2 Random Forest trained successfully.


In [86]:
rf_log_pred = rf_regressor.predict(X2_val)

print("Stage-2 Random Forest")
print("MAE (log):",
      mean_absolute_error(y2_val, rf_log_pred))
print("RMSE (log):",
      np.sqrt(mean_squared_error(y2_val, rf_log_pred)))
print("R²:",
      r2_score(y2_val, rf_log_pred))

Stage-2 Random Forest
MAE (log): 0.7293136520143474
RMSE (log): 0.8994659379269805
R²: -0.03783112626087903


In [87]:
actual_delay_days = np.expm1(y2_val)
predicted_delay_days = np.expm1(rf_log_pred)

print("Performance in actual days")
print("MAE (days):",
      mean_absolute_error(
          actual_delay_days,
          predicted_delay_days
      ))

print("RMSE (days):",
      np.sqrt(
          mean_squared_error(
              actual_delay_days,
              predicted_delay_days
          )
      ))

print("R²:",
      r2_score(
          actual_delay_days,
          predicted_delay_days
      ))

Performance in actual days
MAE (days): 5.762977806664056
RMSE (days): 9.437838945166517
R²: -0.007941144664354471


In [88]:
def delay_to_severity(days):
    if days <= 3:
        return "low"
    elif days <= 7:
        return "moderate"
    elif days <= 14:
        return "high"
    else:
        return "critical"

In [89]:
actual_severity = pd.Series(
    actual_delay_days,
    index=y2_val.index
).apply(delay_to_severity)

predicted_severity = pd.Series(
    predicted_delay_days,
    index=y2_val.index
).apply(delay_to_severity)

severity_accuracy = (
    actual_severity == predicted_severity
).mean()

print(
    "Severity classification agreement:",
    severity_accuracy
)

Severity classification agreement: 0.2776831345826235


In [90]:
timing_features = delivered_orders[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_estimated_delivery_date"
    ]
].copy()

timing_features["promised_delivery_days"] = (
    timing_features["order_estimated_delivery_date"]
    - timing_features["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 3600)

print(
    timing_features[
        [
            "promised_delivery_days"
        ]
    ].describe()
)

       promised_delivery_days
count            96478.000000
mean                23.736347
std                  8.760765
min                  2.008009
25%                 18.330081
50%                 23.230880
75%                 28.407590
max                155.135463


In [91]:
print(
    "Missing promised days:",
    timing_features["promised_delivery_days"].isna().sum()
)

print(
    "Non-positive promised days:",
    (
        timing_features["promised_delivery_days"] <= 0
    ).sum()
)

Missing promised days: 0
Non-positive promised days: 0


In [92]:
timing_features = timing_features.merge(
    shipping_features[
        [
            "order_id",
            "avg_shipping_window_hours",
            "min_shipping_window_hours"
        ]
    ],
    on="order_id",
    how="left",
    validate="one_to_one"
)

timing_features["shipping_window_ratio"] = (
    timing_features["avg_shipping_window_hours"]
    / (timing_features["promised_delivery_days"] * 24)
)

print(
    timing_features[
        [
            "promised_delivery_days",
            "avg_shipping_window_hours",
            "shipping_window_ratio"
        ]
    ].describe()
)

       promised_delivery_days  avg_shipping_window_hours  shipping_window_ratio
count            96478.000000               96478.000000           96478.000000
mean                23.736347                 157.619047               0.303323
std                  8.760765                 111.738309               0.153402
min                  2.008009                  48.101667               0.031869
25%                 18.330081                 120.152222               0.204464
50%                 23.230880                 144.279306               0.273065
75%                 28.407590                 171.058056               0.363084
max                155.135463               25248.108889               7.510913


In [93]:
enhanced_model_base = model_base.merge(
    timing_features[
        [
            "order_id",
            "promised_delivery_days",
            "shipping_window_ratio"
        ]
    ],
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(enhanced_model_base))
print(
    "Unique orders:",
    enhanced_model_base["order_id"].nunique()
)

Rows: 96478
Unique orders: 96478


In [94]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Sort chronologically
enhanced_model_base = (
    enhanced_model_base
    .sort_values("order_purchase_timestamp")
    .reset_index(drop=True)
)

# Temporal split
n = len(enhanced_model_base)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

enh_train = enhanced_model_base.iloc[:train_end].copy()
enh_val = enhanced_model_base.iloc[train_end:val_end].copy()
enh_test = enhanced_model_base.iloc[val_end:].copy()

DROP = [
    "late_delivery",
    "order_id",
    "order_purchase_timestamp"
]

X_train = enh_train.drop(columns=DROP)
y_train = enh_train["late_delivery"]

X_val = enh_val.drop(columns=DROP)
y_val = enh_val["late_delivery"]

X_test = enh_test.drop(columns=DROP)
y_test = enh_test["late_delivery"]

# Feature types
num_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

cat_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# Preprocessing
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features)
])

# Model
stage1_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

stage1_model.fit(X_train, y_train)

# Validation
pred = stage1_model.predict(X_val)
prob = stage1_model.predict_proba(X_val)[:, 1]

print("STAGE 1 — ENHANCED MODEL")
print("Accuracy :", accuracy_score(y_val, pred))
print("Precision:", precision_score(y_val, pred, zero_division=0))
print("Recall   :", recall_score(y_val, pred, zero_division=0))
print("F1       :", f1_score(y_val, pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_val, prob))
print("PR-AUC   :", average_precision_score(y_val, prob))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, pred))


C:\Users\anant\AppData\Local\Temp\ipykernel_26908\4207387607.py:52: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X_train.select_dtypes(


STAGE 1 — ENHANCED MODEL
Accuracy : 0.7009397457158651
Precision: 0.11198428290766209
Recall   : 0.6636481241914618
F1       : 0.19163242435562197
ROC-AUC  : 0.7658860662249829
PR-AUC   : 0.15054653507663882

Confusion Matrix:
[[9631 4068]
 [ 260  513]]


In [95]:
thresholds = np.arange(0.10, 0.91, 0.05)

rows = []

for t in thresholds:
    p = (prob >= t).astype(int)

    rows.append({
        "threshold": round(t, 2),
        "precision": precision_score(y_val, p, zero_division=0),
        "recall": recall_score(y_val, p, zero_division=0),
        "f1": f1_score(y_val, p, zero_division=0),
        "flag_rate": p.mean()
    })

threshold_table = pd.DataFrame(rows)

threshold_table.sort_values(
    "f1",
    ascending=False
).head(5)

,threshold,precision,recall,f1,flag_rate
12,0.70,0.208087,0.272962,0.236150,0.070066
11,0.65,0.174505,0.364812,0.236082,0.111664
10,0.60,0.152784,0.457956,0.229126,0.160102
9,0.55,0.133045,0.556274,0.214732,0.223328
8,0.50,0.111984,0.663648,0.191632,0.316542


In [96]:
stage1_metrics = {
    "accuracy": accuracy_score(y_val, pred),
    "precision": precision_score(y_val, pred, zero_division=0),
    "recall": recall_score(y_val, pred, zero_division=0),
    "f1": f1_score(y_val, pred, zero_division=0),
    "roc_auc": roc_auc_score(y_val, prob),
    "pr_auc": average_precision_score(y_val, prob)
}

stage1_metrics

{'accuracy': 0.7009397457158651,
 'precision': 0.11198428290766209,
 'recall': 0.6636481241914618,
 'f1': 0.19163242435562197,
 'roc_auc': 0.7658860662249829,
 'pr_auc': 0.15054653507663882}